# FHIR Condition Silver Lakeflow Transformation

## Purpose

I use this notebook as the transformation source for the FHIR Silver Lakeflow
pipeline.

I keep the Condition Bronze-to-Silver transformation logic that I previously
developed and tested, while Lakeflow now manages the Silver output and applies
the approved Condition quality contract during refresh.

### Source

`health_insurance.bronze.fhir_condition_raw`

### Pipeline target

`health_insurance.silver.fhir_condition`

### Production changes

In this pipeline version:

- I keep the null-safe FHIR parsing and conformance logic I already validated.
- I use an explicit minimal FHIR Condition schema instead of inferring the
  schema at runtime.
- I retain FHIR `meta.versionId` and `meta.lastUpdated` as technical metadata.
- I keep the latest available version of each Condition resource.
- I load WARN, DROP, and FAIL rules dynamically from Unity Catalog.
- I let Lakeflow manage Silver persistence and quality metrics.

The FHIR Bronze table is cumulative and incrementally populated by Auto Loader.
I therefore use a Lakeflow materialized view over the current Bronze state so I
can resolve the latest Condition version deterministically.


In [0]:
# importing the Lakeflow API, Spark functions, schemas, and window support.

from pyspark import pipelines as dp
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

CATALOG = "health_insurance"

SOURCE_TABLE = f"{CATALOG}.bronze.fhir_condition_raw"
QUALITY_RULES_TABLE = f"{CATALOG}.governance.quality_rules"


In [0]:
# loading the active Condition quality contract from Unity Catalog.

def get_quality_rules(dataset, severity):
    rows = (
        spark.read
        .table(QUALITY_RULES_TABLE)
        .filter(
            (F.col("dataset") == dataset)
            & (F.col("severity") == severity)
            & F.col("is_active")
        )
        .select(
            "rule_name",
            "constraint"
        )
        .collect()
    )

    return {
        row["rule_name"]: row["constraint"]
        for row in rows
    }


CONDITION_WARN_RULES = get_quality_rules("condition", "WARN")
CONDITION_DROP_RULES = get_quality_rules("condition", "DROP")
CONDITION_FAIL_RULES = get_quality_rules("condition", "FAIL")


## Explicit FHIR Condition schema

During development I inferred the combined Condition schema from Bronze.

For the pipeline version, I define only the FHIR structures that my Silver
transformation consumes. This makes the production transformation more
deterministic while still allowing optional FHIR fields to remain NULL.

I also include the FHIR resource metadata required for latest-version handling.


In [0]:
# defining the minimal FHIR Condition schema required by this transformation.

coding_schema = T.StructType([
    T.StructField("system", T.StringType(), True),
    T.StructField("code", T.StringType(), True),
    T.StructField("display", T.StringType(), True)
])

codeable_concept_schema = T.StructType([
    T.StructField(
        "coding",
        T.ArrayType(coding_schema),
        True
    )
])

reference_schema = T.StructType([
    T.StructField("reference", T.StringType(), True)
])

meta_schema = T.StructType([
    T.StructField("versionId", T.StringType(), True),
    T.StructField("lastUpdated", T.StringType(), True)
])

condition_schema = T.StructType([
    T.StructField("id", T.StringType(), True),
    T.StructField(
        "clinicalStatus",
        codeable_concept_schema,
        True
    ),
    T.StructField(
        "verificationStatus",
        codeable_concept_schema,
        True
    ),
    T.StructField(
        "code",
        codeable_concept_schema,
        True
    ),
    T.StructField(
        "subject",
        reference_schema,
        True
    ),
    T.StructField(
        "encounter",
        reference_schema,
        True
    ),
    T.StructField("onsetDateTime", T.StringType(), True),
    T.StructField("abatementDateTime", T.StringType(), True),
    T.StructField("recordedDate", T.StringType(), True),
    T.StructField("meta", meta_schema, True)
])


## Reusable Condition transformation

I keep the transformation itself separate from the Lakeflow dataset definition.

The function performs the same Condition parsing, null-safe coding extraction,
Patient and Encounter reference parsing, timestamp conversion, duration
derivation, and status standardization I validated in the development notebook.

I additionally retain FHIR version metadata and resolve the latest resource
version before publishing Silver.


In [0]:
#  applying the validated FHIR Condition Bronze-to-Silver transformation.

def transform_condition(condition_bronze_df):

    condition_parsed_df = (
        condition_bronze_df
        .withColumn(
            "_record_hash",
            F.sha2(F.col("raw_json"), 256)
        )
        .withColumn(
            "condition",
            F.from_json(
                F.col("raw_json"),
                condition_schema
            )
        )
    )

    condition_core_df = (
        condition_parsed_df
        .select(
            F.col("condition.id").alias("condition_id"),

            F.col("condition.clinicalStatus").alias(
                "clinical_status_struct"
            ),
            F.col("condition.verificationStatus").alias(
                "verification_status_struct"
            ),
            F.col("condition.code").alias(
                "condition_code_struct"
            ),

            F.col("condition.subject.reference").alias(
                "patient_reference"
            ),
            F.col("condition.encounter.reference").alias(
                "encounter_reference"
            ),

            F.col("condition.onsetDateTime").alias(
                "onset_datetime_raw"
            ),
            F.col("condition.abatementDateTime").alias(
                "abatement_datetime_raw"
            ),
            F.col("condition.recordedDate").alias(
                "recorded_datetime_raw"
            ),

            F.col("condition.meta.versionId").alias(
                "fhir_version_id"
            ),
            F.to_timestamp(
                F.col("condition.meta.lastUpdated")
            ).alias("fhir_last_updated"),

            "_record_hash",
            "_ingested_at",
            "_source_system",
            "_resource_type"
        )
    )

    condition_status_df = (
        condition_core_df

        .withColumn(
            "clinical_status",
            F.expr(
                "get(clinical_status_struct.coding.code, 0)"
            )
        )

        .withColumn(
            "verification_status",
            F.expr(
                "get(verification_status_struct.coding.code, 0)"
            )
        )
    )

    condition_coded_df = (
        condition_status_df

        .withColumn(
            "condition_code",
            F.expr(
                "get(condition_code_struct.coding.code, 0)"
            )
        )

        .withColumn(
            "condition_name",
            F.expr(
                "get(condition_code_struct.coding.display, 0)"
            )
        )

        .withColumn(
            "code_system",
            F.expr(
                "get(condition_code_struct.coding.system, 0)"
            )
        )
    )

    condition_refs_df = (
        condition_coded_df

        .withColumn(
            "patient_id",
            F.when(
                F.col("patient_reference").startswith("Patient/"),
                F.regexp_extract(
                    F.col("patient_reference"),
                    r"^Patient/(.+)$",
                    1
                )
            ).otherwise(
                F.lit(None).cast("string")
            )
        )

        .withColumn(
            "encounter_id",
            F.when(
                F.col("encounter_reference").startswith("Encounter/"),
                F.regexp_extract(
                    F.col("encounter_reference"),
                    r"^Encounter/(.+)$",
                    1
                )
            ).otherwise(
                F.lit(None).cast("string")
            )
        )
    )

    condition_typed_df = (
        condition_refs_df

        .withColumn(
            "onset_datetime",
            F.to_timestamp("onset_datetime_raw")
        )

        .withColumn(
            "abatement_datetime",
            F.to_timestamp("abatement_datetime_raw")
        )

        .withColumn(
            "recorded_datetime",
            F.to_timestamp("recorded_datetime_raw")
        )
    )

    condition_enriched_df = (
        condition_typed_df

        .withColumn(
            "condition_duration_days",
            F.when(
                F.col("onset_datetime").isNotNull()
                & F.col("abatement_datetime").isNotNull(),
                F.datediff(
                    F.to_date("abatement_datetime"),
                    F.to_date("onset_datetime")
                )
            ).otherwise(
                F.lit(None).cast("int")
            )
        )
    )

    condition_standardized_df = (
        condition_enriched_df

        .withColumn(
            "clinical_status",
            F.upper(F.trim("clinical_status"))
        )

        .withColumn(
            "verification_status",
            F.upper(F.trim("verification_status"))
        )
    )

    condition_conformed_df = (
        condition_standardized_df

        .select(
            "condition_id",
            "patient_id",
            "encounter_id",

            "condition_code",
            "condition_name",
            "code_system",

            "clinical_status",
            "verification_status",

            "onset_datetime",
            "abatement_datetime",
            "recorded_datetime",
            "condition_duration_days",

            "fhir_version_id",
            "fhir_last_updated",

            "_record_hash",
            "_source_system",
            "_resource_type",
            "_ingested_at"
        )

        .withColumn(
            "_silver_transformed_at",
            F.current_timestamp()
        )
    )

    # I am using the FHIR Condition ID as the main deduplication key.
    # If an ID is missing, I fall back to the raw payload hash so malformed
    # records remain separate and can still be evaluated by DROP expectations.

    condition_versioned_df = (
        condition_conformed_df
        .withColumn(
            "_dedup_key",
            F.coalesce(
                F.col("condition_id"),
                F.col("_record_hash")
            )
        )
    )

    latest_condition_window = (
        Window
        .partitionBy("_dedup_key")
        .orderBy(
            F.col("fhir_last_updated").desc_nulls_last(),
            F.col("_ingested_at").desc_nulls_last(),
            F.col("fhir_version_id").desc_nulls_last()
        )
    )

    condition_latest_df = (
        condition_versioned_df

        .withColumn(
            "_version_rank",
            F.row_number().over(latest_condition_window)
        )

        .filter(
            F.col("_version_rank") == 1
        )

        .drop(
            "_version_rank",
            "_dedup_key",
            "_record_hash"
        )
    )

    return condition_latest_df


## Lakeflow-managed Condition Silver dataset

I attach the centrally governed Condition expectations directly to the Silver
materialized view.

- WARN rules preserve the record and expose quality metrics.
- DROP rules remove unusable Condition records from validated Silver.
- FAIL rules stop the Condition flow when a critical contract is violated.

The materialized view reads the cumulative Bronze state, resolves the latest
FHIR Condition version, and publishes one current Silver record per Condition.


In [0]:
# defining the pipeline-managed FHIR Condition Silver materialized view.

@dp.materialized_view(
    name="fhir_condition",
    comment="Validated and latest-version FHIR Condition records."
)
@dp.expect_all(CONDITION_WARN_RULES)
@dp.expect_all_or_drop(CONDITION_DROP_RULES)
@dp.expect_all_or_fail(CONDITION_FAIL_RULES)
def fhir_condition():

    condition_bronze_df = spark.read.table(
        SOURCE_TABLE
    )

    return transform_condition(
        condition_bronze_df
    )


## Pipeline result

This notebook no longer writes `health_insurance.silver.fhir_condition`
manually.

When I add it as a source to the FHIR Silver Lakeflow pipeline:

1. Lakeflow reads the cumulative FHIR Condition Bronze table.
2. I parse the Condition JSON with an explicit production schema.
3. I apply the previously validated Condition transformation logic.
4. I retain FHIR version metadata and select the latest resource version.
5. Lakeflow evaluates the active governed Condition quality rules.
6. Lakeflow manages the Silver materialized view and expectation metrics.

The development-only schema inference, `display()`, profiling counts, direct
Delta overwrite, row reconciliation, and post-write verification cells are no
longer part of the pipeline execution path.
